# Module 4: AI Recommendation Model (ML-Based)

## Objective
Train machine learning models to predict:
- Packaging cost (INR)
- CO₂ impact (kg)

Then generate a material ranking system using ML predictions + sustainability metrics.

## Models
- RandomForestRegressor → cost prediction
- XGBoostRegressor → CO₂ prediction

## Metrics
- RMSE
- MAE
- R² Score


In [1]:
import pandas as pd

In [2]:
import numpy as np


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor


In [4]:
materials_df = pd.read_csv("../data/processed/materials_dataset.csv")
products_df  = pd.read_csv("../data/processed/products_dataset.csv")

materials_df["_key"] = 1
products_df["_key"] = 1
ml_df = products_df.merge(materials_df, on="_key").drop(columns=["_key"])

# Targets (same as Module 3)
ml_df["target_cost_inr"] = ml_df["cost_per_unit_inr"] + (ml_df["product_weight_kg"] * 10)
ml_df["target_co2_kg"]   = ml_df["co2_emission_kg"] + (ml_df["product_weight_kg"] * 0.5)

ml_features = [
    "product_weight_kg",
    "required_strength_score",
    "preferred_biodegradability_score",
    "strength_score",
    "weight_capacity_kg",
    "biodegradability_score",
    "recyclability_percent",
    "co2_emission_kg",
    "cost_per_unit_inr"
]

X = ml_df[ml_features].copy()
y_cost = ml_df["target_cost_inr"].copy()
y_co2  = ml_df["target_co2_kg"].copy()

print("ML dataset:", ml_df.shape)
print("X:", X.shape, "| y_cost:", y_cost.shape, "| y_co2:", y_co2.shape)


ML dataset: (21000, 21)
X: (21000, 9) | y_cost: (21000,) | y_co2: (21000,)


In [5]:
X_train, X_test, y_cost_train, y_cost_test = train_test_split(
    X, y_cost, test_size=0.2, random_state=42
)

_, _, y_co2_train, y_co2_test = train_test_split(
    X, y_co2, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (16800, 9) Test: (4200, 9)


In [7]:
rf_cost = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf_cost.fit(X_train, y_cost_train)
pred_cost = rf_cost.predict(X_test)

rmse_cost = mean_squared_error(y_cost_test, pred_cost) ** 0.5
mae_cost  = mean_absolute_error(y_cost_test, pred_cost)
r2_cost   = r2_score(y_cost_test, pred_cost)

print("COST MODEL (RandomForest)")
print("RMSE:", round(rmse_cost, 4))
print("MAE :", round(mae_cost, 4))
print("R2  :", round(r2_cost, 4))


COST MODEL (RandomForest)
RMSE: 0.4799
MAE : 0.1231
R2  : 0.9999


In [9]:
try:
    from xgboost import XGBRegressor
    print("xgboost is installed")
except Exception as e:
    print("xgboost not installed:", e)


xgboost is installed


In [11]:
from xgboost import XGBRegressor

xgb_co2 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_co2.fit(X_train_scaled, y_co2_train)
pred_co2 = xgb_co2.predict(X_test_scaled)

rmse_co2 = mean_squared_error(y_co2_test, pred_co2) ** 0.5
mae_co2  = mean_absolute_error(y_co2_test, pred_co2)
r2_co2   = r2_score(y_co2_test, pred_co2)

print("CO2 MODEL (XGBoost)")
print("RMSE:", round(rmse_co2, 4))
print("MAE :", round(mae_co2, 4))
print("R2  :", round(r2_co2, 4))


CO2 MODEL (XGBoost)
RMSE: 0.017
MAE : 0.0093
R2  : 1.0


In [12]:
# Predict cost and CO2 for full dataset
ml_df["predicted_cost_inr"] = rf_cost.predict(X)
ml_df["predicted_co2_kg"]   = xgb_co2.predict(scaler.transform(X))

ml_df[["material_name", "product_name", "predicted_cost_inr", "predicted_co2_kg"]].head()


,material_name,product_name,predicted_cost_inr,predicted_co2_kg
0,Single-wall corrugated cardboard,Smartphone,57.195333,1.707240
1,Double-wall corrugated cardboard,Smartphone,77.200000,2.111783
2,Triple-wall corrugated cardboard,Smartphone,97.200000,2.608674
3,Kraft linerboard,Smartphone,52.198333,1.608071
4,Test linerboard,Smartphone,50.196667,1.912408


In [13]:
ml_df["cost_score"] = 1 - (
    ml_df["predicted_cost_inr"] / ml_df["predicted_cost_inr"].max()
)

ml_df["co2_score"] = 1 - (
    ml_df["predicted_co2_kg"] / ml_df["predicted_co2_kg"].max()
)


In [14]:
ml_df["final_ai_score"] = (
    0.4 * ml_df["cost_score"] +
    0.4 * ml_df["co2_score"] +
    0.2 * (ml_df["biodegradability_score"] / 10)
)


In [15]:
selected_product = "Smartphone"

top_materials = (
    ml_df[ml_df["product_name"] == selected_product]
    .sort_values("final_ai_score", ascending=False)
    .head(10)
)

top_materials[
    ["material_name", "predicted_cost_inr", "predicted_co2_kg", "final_ai_score"]
]


,material_name,predicted_cost_inr,predicted_co2_kg,final_ai_score
10,Thin-wall molded pulp,44.199333,1.114486,0.944284
9,Thick-wall molded pulp,47.198333,1.310894,0.938086
29,Sugarcane bagasse pulp,62.200000,1.210126,0.928765
22,Areca leaf plates,67.200000,1.210503,0.924978
113,Tamper-evident paper seals,42.200000,1.307070,0.921940
32,Palm leaf packaging,72.200000,1.307614,0.919255
3,Kraft linerboard,52.198333,1.608071,0.908358
30,Bamboo molded pulp,87.200000,1.412600,0.905816
117,Compostable takeaway clamshells,92.200000,1.311374,0.904064
114,Inflatable paper air pillows,62.200000,1.506896,0.902824


In [17]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(rf_cost, models_dir / "rf_cost_model.pkl")
joblib.dump(xgb_co2, models_dir / "xgb_co2_model.pkl")
joblib.dump(scaler,  models_dir / "scaler.pkl")

print("Models saved in /models folder")


Models saved in /models folder


## Module 4 Completion Summary 

- Trained Random Forest Regressor for packaging cost prediction
- Trained XGBoost Regressor for CO₂ footprint prediction
- Evaluated models using RMSE, MAE, and R² (high accuracy achieved)
- Generated ML-based cost and CO₂ predictions for all product–material pairs
- Designed an AI material ranking system using normalized predictions
- Produced Top-N material recommendations for selected products

Module 4 successfully implements AI-driven sustainable packaging recommendations.
